In [34]:
from srlearn.rdn import BoostedRDNClassifier
from srlearn import Background, Database

def get_mulval_like_data():

    train = Database()

    train.modes = [
        "exec_code(+host, #priv).",
        "hacl(+host1, +host2).",
        "vul_exists(+host, -vul, -program).",
        "vul_property(+vul, #type, #impact).",
        "client_program(+host, +program, -priv).",
    ]

    # =====================================================
    # POSITIVOS (vários!)
    # =====================================================
    train.pos = [
        "exec_code(h1, root).",
        "exec_code(h9, root).",
        "exec_code(h10, root).",
        "exec_code(h11, root).",
        "exec_code(h12, root).",
    ]

    # =====================================================
    # NEGATIVOS
    # =====================================================
    train.neg = [
        "exec_code(h2, root).",
        "exec_code(h3, root).",
        "exec_code(h4, root).",
        "exec_code(h5, root).",
        "exec_code(h6, root).",
        "exec_code(h7, root).",
        "exec_code(h8, root).",
        "exec_code(h13, root).",
        "exec_code(h14, root).",
        "exec_code(h15, root).",
    ]

    train.facts = [

        # =================================================
        # EXECUÇÃO INICIAL EM MÚLTIPLOS HOSTS
        # =================================================
        "exec_code(a1, user).",
        "exec_code(a2, user).",
        "exec_code(a3, user).",

        # =================================================
        # POSITIVOS CORRETOS
        # =================================================
        "hacl(a1, h1).",
        "vul_exists(h1, v1, p1).",
        "vul_property(v1, remoteExploit, privEscalation).",
        "client_program(h1, p1, user).",

        "hacl(a1, h9).",
        "vul_exists(h9, v9, p9).",
        "vul_property(v9, remoteExploit, privEscalation).",
        "client_program(h9, p9, user).",

        "hacl(a2, h10).",
        "vul_exists(h10, v10, p10).",
        "vul_property(v10, remoteExploit, privEscalation).",
        "client_program(h10, p10, user).",

        "hacl(a2, h11).",
        "vul_exists(h11, v11, p11).",
        "vul_property(v11, remoteExploit, privEscalation).",
        "client_program(h11, p11, user).",

        "hacl(a3, h12).",
        "vul_exists(h12, v12, p12).",
        "vul_property(v12, remoteExploit, privEscalation).",
        "client_program(h12, p12, user).",

        # =================================================
        # NEGATIVOS ESTRUTURAIS
        # =================================================

        # tipo errado
        "hacl(a1, h2).",
        "vul_exists(h2, v2, p2).",
        "vul_property(v2, localExploit, privEscalation).",
        "client_program(h2, p2, user).",

        # impacto errado
        "hacl(a1, h3).",
        "vul_exists(h3, v3, p3).",
        "vul_property(v3, remoteExploit, dos).",
        "client_program(h3, p3, user).",

        # sem acesso
        "vul_exists(h4, v4, p4).",
        "vul_property(v4, remoteExploit, privEscalation).",
        "client_program(h4, p4, user).",

        # sem vuln
        "hacl(a1, h5).",

        # sem exec inicial
        "hacl(b1, h6).",
        "vul_exists(h6, v6, p6).",
        "vul_property(v6, remoteExploit, privEscalation).",
        "client_program(h6, p6, user).",

        # mistura variada
        "vul_exists(h7, v7, p7).",
        "vul_property(v7, localExploit, dos).",

        "hacl(a3, h8).",
        "vul_exists(h8, v8, p8).",
        "vul_property(v8, localExploit, dos).",

        "hacl(a3, h13).",

        "vul_exists(h14, v14, p14).",

        "hacl(b2, h15).",
        "vul_exists(h15, v15, p15).",
        "vul_property(v15, remoteExploit, privEscalation).",
    ]

    return train


train = get_mulval_like_data()
bk = Background(modes=train.modes, recursion=True)

clf = BoostedRDNClassifier(
    background=bk,
    target="exec_code",
    max_tree_depth=8,
    node_size=1,
    n_estimators=250
)

clf.fit(train)

print("\nTreinamento concluído com múltiplos cenários estruturais.")


/Users/carlosesb/Documents/test/.venv/lib/python3.9/site-packages/srlearn/bsrl_data/data1/train

Treinamento concluído com múltiplos cenários estruturais.


In [4]:
import numpy as np
from srlearn.rdn import BoostedRDNClassifier
from srlearn import Background, Database


# ==========================================
# DATASET ESTILO MULVAL (MAIOR)
# ==========================================
def get_mulval_like_data():

    train = Database.from_files('dataset/fold00/train/pos.pl', 'dataset/fold00/train/neg.pl', 'dataset/fold00/train/facts.pl')

    train.modes = [
        "exec_code(+attacker, +host, +priv).",
        "vul_exists(+host, -vul, -program).",
        "vul_property(+vul, #type, #impact).",
        "client_program(+host, +program, -priv).",
        "malicious(+attacker)."
    ]

    return train


# ==========================================
# TREINAMENTO
# ==========================================
train = get_mulval_like_data()
bk = Background(modes=train.modes)

clf = BoostedRDNClassifier(
    background=bk,
    target="exec_code",
    max_tree_depth=6,
    node_size=4,
    n_estimators=80
)

clf.fit(train)

print("\nTreinamento concluído com dataset ampliado.")

/Users/carlosesb/Documents/test/.venv/lib/python3.9/site-packages/srlearn/bsrl_data/data3/train

Treinamento concluído com dataset ampliado.


In [3]:
# ==========================================
# TESTE
# ==========================================
test = Database()
test.modes = train.modes

test.pos = ["exec_code(att1, t1, user)."]
test.neg = ["exec_code(att1, t2, user)."]

test.facts = [

    # atacante malicioso
    "malicious(att1).",

    # -------------------------
    # t1 -> deve ser POSITIVO
    # -------------------------
    "vul_exists(t1, vt1, pt1).",
    "vul_property(vt1, remoteExploit, privEscalation).",
    "client_program(t1, pt1, user).",

    # -------------------------
    # t2 -> deve ser NEGATIVO
    # (não é remoteExploit)
    # -------------------------
    "vul_exists(t2, vt2, pt2).",
    "vul_property(vt2, localExploit, privEscalation).",
    "client_program(t2, pt2, user)."
]

probs = clf.predict_proba(test)

print("\n=== TEST RESULTS ===")
print(f"t1 probability (expected HIGH): {probs[0]:.4f}")
print(f"t2 probability (expected LOW):  {probs[1]:.4f}")


=== TEST RESULTS ===
t1 probability (expected HIGH): 0.9870
t2 probability (expected LOW):  0.0117


In [14]:
import re
from pathlib import Path

# ==========================================
# NORMALIZAR IP (WILL SAFE)
# ==========================================
def normalize_ip(ip):
    """
    Converte:
    146.164.209.52
    ->
    h146_164_209_52
    """
    ip = ip.strip().strip('"')
    return "h" + ip.replace(".", "_")

# ==========================================
# CARREGAR FATOS DO ARQUIVO
# ==========================================
def load_prolog_facts(filepath):
    facts = []

    ip_pattern = re.compile(r'\b\d+\.\d+\.\d+\.\d+\b')

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("%"):
                continue

            # substituir todos os IPs encontrados
            def replace_ip(match):
                return normalize_ip(match.group(0))

            line = ip_pattern.sub(replace_ip, line)
            facts.append(line)

    return facts

# ==========================================
# EXTRAIR HOSTS DE client_program/3
# ==========================================
def extract_hosts(facts):
    hosts = set()
    pattern = re.compile(r"client_program\(([^,]+),")
    for fact in facts:
        match = pattern.search(fact)
        if match:
            host = match.group(1).strip()
            hosts.add(host)
    return sorted(hosts)

# ==========================================
# MAIN
# ==========================================
facts = load_prolog_facts("hosts.pl")
hosts = extract_hosts(facts)

# IPs positivos
pos_hosts_raw = ["146.164.93.169", "146.164.34.82"]
pos_hosts = [normalize_ip(ip) for ip in pos_hosts_raw]

# Hosts negativos = todos exceto os positivos
neg_hosts = [h for h in hosts if h not in pos_hosts]

print("\n=== PROBABILIDADE DE EXEC_CODE (POSITIVOS E NEGATIVOS) ===\n")

test = Database()
test.modes = train.modes

# Adiciona positivos
test.pos = [f"execCode(att1, {host}, root)." for host in pos_hosts]

# Adiciona negativos
test.neg = [f"execCode(att1, {host}, root)." for host in neg_hosts]

# Adiciona fatos
test.facts = ["attackerLocated(internet)."] + facts

# Calcula probabilidades
probs = clf.predict_proba(test)

# Mostra resultados
for host, prob in zip(pos_hosts + neg_hosts, probs):
    print(f"Host {host} -> {prob:.4f}")


=== PROBABILIDADE DE EXEC_CODE (POSITIVOS E NEGATIVOS) ===



Exception in thread "main" edu.wisc.cs.will.Utils.WILLthrownError: 
 #args do not match!  TargetArgSpecs = [D[+host], E[+privilege]] while ex = execCode(att1, h146_164_93_169, root)
	at edu.wisc.cs.will.Utils.Utils.error(Utils.java:484)
	at edu.wisc.cs.will.DataSetUtils.TypeManagement.recordTypedConstantsFromTheseExamples(TypeManagement.java:346)
	at edu.wisc.cs.will.DataSetUtils.TypeManagement.collectTypedConstants(TypeManagement.java:87)
	at edu.wisc.cs.will.ILP.LearnOneClause.initialize(LearnOneClause.java:1298)
	at edu.wisc.cs.will.ILP.ILPouterLoop.initialize(ILPouterLoop.java:1360)
	at edu.wisc.cs.will.Boosting.RDN.WILLSetup.setup(WILLSetup.java:262)
	at edu.wisc.cs.will.Boosting.Common.RunBoostedModels.setupWILLForTest(RunBoostedModels.java:183)
	at edu.wisc.cs.will.Boosting.Common.RunBoostedModels.inferModel(RunBoostedModels.java:159)
	at edu.wisc.cs.will.Boosting.RDN.RunBoostedRDN.runJob(RunBoostedRDN.java:49)
	at edu.wisc.cs.will.Boosting.Common.RunBoostedModels.main(RunBooste

RuntimeError: Error when running shell command: java -jar /Users/carlosesb/Documents/test/.venv/lib/python3.9/site-packages/srlearn/BoostSRL.jar -i -test /Users/carlosesb/Documents/test/.venv/lib/python3.9/site-packages/srlearn/bsrl_data/data0/test -model /Users/carlosesb/Documents/test/.venv/lib/python3.9/site-packages/srlearn/bsrl_data/data0/train/models -target execCode -trees 10 -aucJarPath /Users/carlosesb/Documents/test/.venv/lib/python3.9/site-packages/srlearn > /Users/carlosesb/Documents/test/.venv/lib/python3.9/site-packages/srlearn/bsrl_data/data0/test_output.txt

In [11]:
from graphviz import Source

with open("/Users/carlosesb/Documents/test/.venv/lib/python3.9/site-packages/srlearn/bsrl_data/data3/train/models/bRDNs/dotFiles/CombinedTreesexec_code.dot") as f:
    dot_graph = f.read()

graph = Source(dot_graph)
graph.render("tree", format="png", view=True)

/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/subprocess.py:1052: ResourceWarning: subprocess 96125 is still running
  _warn("subprocess %s is still running" % self.pid,


'tree.png'

## Shodan

In [13]:
import json
import re

INPUT_FILE = "ShodanUFRJ.jsonl"
OUTPUT_FILE = "hosts.pl"


def sanitize_program_name(name, version):
    """
    Converte nome+versão em átomo Prolog seguro.
    Ex: OpenSSH 8.7 -> openssh_8_7
    """
    base = f"{name}_{version}"
    base = base.lower()
    base = re.sub(r"[^a-z0-9]+", "_", base)
    base = re.sub(r"_+", "_", base)
    return base.strip("_")


def detect_privilege(port):
    """
    Define privilégio baseado na exposição.
    Porta aberta na rede -> remote
    (Você pode adaptar se quiser algo mais sofisticado)
    """
    return "remote"


with open(INPUT_FILE, "r", encoding="utf-8") as infile, \
     open(OUTPUT_FILE, "w", encoding="utf-8") as outfile:

    for line in infile:
        data = json.loads(line)

        host = data["ip_str"]
        port = data.get("port")
        version = data.get("version", "")
        transport = data.get("transport", "")

        # Nome do programa (ex: ssh ou serviço detectado)
        program_name = data.get("product", transport or "service")
        program = sanitize_program_name(program_name, version)

        privilege = detect_privilege(port)

        # 1️⃣ client_program(+host, +program, -priv).
        outfile.write(
            f"client_program({host}, {program}, {privilege}).\n"
        )

        # 2️⃣ vul_exists(+host, -vul, -program).
        vulns = data.get("vulns", {})
        for cve in vulns.keys():
            outfile.write(
                f"vul_exists({host}, '{cve}', {program}).\n"
            )

print("Arquivo Prolog gerado com sucesso:", OUTPUT_FILE)


Arquivo Prolog gerado com sucesso: hosts.pl


In [ ]:
import json

INPUT_FILE = "ShodanUFRJ.jsonl"

all_vulns = set()

with open(INPUT_FILE, "r", encoding="utf-8") as infile:
    for line in infile:
        data = json.loads(line)
        vulns = data.get("vulns", {})
        
        for cve in vulns.keys():
            all_vulns.add(cve)

all_vulns = list(all_vulns)

print(len(all_vulns))

1008


In [ ]:
import requests
import time
import re

# =========================
# CONFIG
# =========================

API_KEY = "90571d7f-430e-440f-aed3-b9454fb7eba5"
NVD_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"
OUTPUT_FILE = "vul_properties.pl"

cve_list = all_vulns

FLAGS = re.IGNORECASE


# =========================
# VOCABULARIO EXPANDIDO
# =========================

def compile_patterns(patterns):
    return [re.compile(p, FLAGS) for p in patterns]


vocab_priv_escalation = compile_patterns([
    r'gain root',
    r'gain root access',
    r'root shell access',
    r'obtain root',
    r'execute as root',
    r'run as root',
    r'gain privilege',
    r'gain privileges',
    r'elevate privilege',
    r'elevate privileges',
    r'elevation of privilege',
    r'escalation of privilege',
    r'privilege escalation',
    r'become root',
    r'become administrator',
    r'gain admin',
    r'obtain admin',
    r'gain administrator'
])

# =========================
# CLASSIFICADORES
# =========================

def match_any(vocab, text):
    return any(pattern.search(text) for pattern in vocab)


def classify_attack_vector(metrics):
    attack_vector = None

    for metric in metrics:
        cvss = metric.get("cvssData", {})
        attack_vector = cvss.get("attackVector")
        if attack_vector:
            break

    if attack_vector in ["NETWORK", "ADJACENT"]:
        return "remoteExploit"
    else:
        return "localExploit"


def classify_impact(description):
    desc = re.sub(r'\s+', ' ', description.lower())

    if match_any(vocab_priv_escalation, desc):
        return "privEscalation"

    return "other"


# =========================
# CONSULTA NVD
# =========================

def query_nvd(cve_id):
    headers = {"apiKey": API_KEY}
    params = {"cveId": cve_id}

    response = requests.get(NVD_URL, headers=headers, params=params)
    response.raise_for_status()

    data = response.json()

    if not data.get("vulnerabilities"):
        return None

    cve_data = data["vulnerabilities"][0]["cve"]

    # descrição
    description = ""
    for d in cve_data.get("descriptions", []):
        if d["lang"] == "en":
            description = d["value"]
            break

    # métricas CVSS
    metrics = []
    metrics_data = cve_data.get("metrics", {})

    if "cvssMetricV31" in metrics_data:
        metrics = metrics_data["cvssMetricV31"]
    elif "cvssMetricV30" in metrics_data:
        metrics = metrics_data["cvssMetricV30"]

    return metrics, description


# =========================
# GERAÇÃO DO ARQUIVO PROLOG
# =========================

with open(OUTPUT_FILE, "w", encoding="utf-8") as outfile:

    outfile.write("% Arquivo gerado automaticamente\n\n")

    for cve in cve_list:
        try:
            result = query_nvd(cve)

            if not result:
                print(f"Não encontrado: {cve}")
                continue

            metrics, description = result

            exploit_type = classify_attack_vector(metrics)
            impact = classify_impact(description)

            outfile.write(
                f"vul_property('{cve}', {exploit_type}, {impact}).\n"
            )

            print(f"Processado: {cve}")

            # evitar rate limit NVD
            time.sleep(1.0)

        except Exception as e:
            print(f"Erro ao processar {cve}: {e}")

print(f"\nArquivo salvo em: {OUTPUT_FILE}")

Processado: CVE-2009-1955
Processado: CVE-2014-3570
Processado: CVE-2011-1945
Processado: CVE-2020-13950
Processado: CVE-2016-7417
Processado: CVE-2018-1312
Processado: CVE-2022-3590
Processado: CVE-2010-1915
Processado: CVE-2020-7043
Processado: CVE-2014-0231
Processado: CVE-2022-28330
Processado: CVE-2009-0023
Processado: CVE-2019-0215
Processado: CVE-2024-40898
Processado: CVE-2018-11763
Processado: CVE-2011-4108
Processado: CVE-2016-5768
Processado: CVE-2015-1352
Processado: CVE-2013-0166
Processado: CVE-2022-26488
Processado: CVE-2015-4024
Processado: CVE-2023-44487
Processado: CVE-2015-8865
Processado: CVE-2019-9020
Processado: CVE-2010-0740
Processado: CVE-2009-4418
Processado: CVE-2019-11048
Processado: CVE-2016-10010
Processado: CVE-2023-22622
Processado: CVE-2012-1165
Processado: CVE-2023-0286
Processado: CVE-2011-1092
Processado: CVE-2013-6449
Processado: CVE-2016-4537
Processado: CVE-2008-2168
Processado: CVE-2011-4415
Processado: CVE-2016-7416
Processado: CVE-2015-1351
Pro

## GERAR TREINO PARA TODAS AS REGRAS DE INTERAÇÃO

In [ ]:
import numpy as np
from srlearn.rdn import BoostedRDNClassifier
from srlearn import Background, Database

def get_mulval_like_data():

    train = Database.from_files('dataset/fold05/train/pos.pl', 'dataset/fold05/train/neg.pl', 'dataset/fold05/train/facts.pl')

    train.modes = [
        "execCode(+host, +privilege).", 
        
        "vulExists(+host, -cve, -software).",
        "vulProperty(+cve, #exploitType, #effect).",
        
        "networkServiceInfo(+host, +software, -protocol, +port, -privilege).",
        
        "hacl(+host, -host, -protocol, -port).",
        "hacl(+zone, -host, -protocol, -port).",
        
        "attackerLocated(-host).",
        "attackerLocated(-zone).",
        
        "initialAccess(-host, -privilege)."
    ]

    return train


# ==========================================
# TREINAMENTO
# ==========================================
train = get_mulval_like_data()
bk = Background(modes=train.modes)

clf = BoostedRDNClassifier(
    background=bk,
    target="execCode",
    max_tree_depth=6,
    node_size=3,
    n_estimators=10,
)

clf.fit(train)

print("\nTreinamento concluído com dataset ampliado.")

/Users/carlosesb/Documents/test/.venv/lib/python3.9/site-packages/srlearn/bsrl_data/data4/train

Treinamento concluído com dataset ampliado.


In [20]:
# ==========================================
# TESTE
# ==========================================
test = Database()
test.modes = train.modes

test.pos = ["execCode(146_164_34_82, root)."]
test.neg = ["execCode(146_164_34_83, root)."]

test.facts = [

    "attackerLocated(internet).",

    # conectividade
    "hacl(internet, 146_164_34_82, tcp, 80).",
    "hacl(internet, 146_164_34_83, tcp, 80).",

    # HOST 82 (esperado HIGH)
    "networkServiceInfo(146_164_34_82, grafana_open_source_6_7_4, tcp, 80, root).",
    "vulExists(146_164_34_82, cve_2022_21703, grafana_open_source_6_7_4).",
    "vulProperty(cve_2022_21703, remoteExploit, privEscalation).",

    # HOST 83 (esperado LOW)
    "networkServiceInfo(146_164_34_83, apache_httpd_2_4_29, tcp, 80, root).",
    "vulExists(146_164_34_83, cve_2024_39573, apache_httpd_2_4_29).",
    "vulProperty(cve_2024_39573, remoteExploit, other)."
]

probs = clf.predict_proba(test)

print("\n=== TEST RESULTS ===")
print(f"t1 probability (expected HIGH): {probs[0]:.4f}")
print(f"t2 probability (expected LOW):  {probs[1]:.4f}")


=== TEST RESULTS ===
t1 probability (expected HIGH): 0.5885
t2 probability (expected LOW):  0.0593


In [24]:
from srlearn.rdn import BoostedRDNClassifier
from srlearn import Background
import numpy as np
import csv


def build_test_set(train):

    test = Database()

    test.pos = ["execCode(146_164_34_82, root)."]
    test.neg = ["execCode(146_164_34_83, root)."]

    test.facts = [

        "attackerLocated(internet).",

        # conectividade
        "hacl(internet, 146_164_34_82, tcp, 80).",
        "hacl(internet, 146_164_34_83, tcp, 80).",

        # HOST 82 (esperado HIGH)
        "networkServiceInfo(146_164_34_82, grafana_open_source_6_7_4, tcp, 80, root).",
        "vulExists(146_164_34_82, cve_2022_21703, grafana_open_source_6_7_4).",
        "vulProperty(cve_2022_21703, remoteExploit, privEscalation).",

        # HOST 83 (esperado LOW)
        "networkServiceInfo(146_164_34_83, apache_httpd_2_4_29, tcp, 80, root).",
        "vulExists(146_164_34_83, cve_2024_39573, apache_httpd_2_4_29).",
        "vulProperty(cve_2024_39573, remoteExploit, other)."
    ]

    test.modes = train.modes
    return test

# ==========================================
# TREINAMENTO
# ==========================================
# train = get_mulval_like_data()
# bk = Background(modes=train.modes)

# clf = BoostedRDNClassifier(
#     background=bk,
#     target="execCode",
#     max_tree_depth=6,
#     node_size=3,
#     n_estimators=20,
# )

# clf.fit(train)

test = build_test_set(train)

x = np.arange(1, 10)

with open("convergencia_hosts.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["n_trees", "prob_82", "prob_83"])

    for n_trees in x:
        clf.n_estimators = n_trees
        probs = clf.predict_proba(test)

        writer.writerow([n_trees, probs[0], probs[1]])

print("Arquivo convergencia_hosts.csv salvo.")

Arquivo convergencia_hosts.csv salvo.


In [6]:
import graphviz

# Carrega o arquivo .dot
with open("/Users/carlosesb/Documents/test/.venv/lib/python3.9/site-packages/srlearn/bsrl_data/data4/train/models/bRDNs/dotFiles/CombinedTreesexecCode.dot") as f:
    dot_graph = f.read()

# Renderiza e visualiza
source = graphviz.Source(dot_graph)
source.view()  # Isso abre o PDF gerado no visualizador padrão

/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/subprocess.py:1052: ResourceWarning: subprocess 30613 is still running
  _warn("subprocess %s is still running" % self.pid,


'Source.gv.pdf'

In [3]:
import numpy as np
from srlearn.rdn import BoostedRDNClassifier
from srlearn import Background, Database


# ==========================================
# DATASET ESTILO MULVAL (MAIOR)
# ==========================================
def get_mulval_like_data():

    test = Database.from_files('dataset/fold05/test/pos.pl', 'dataset/fold05/test/neg.pl', 'dataset/fold05/test/facts.pl')

    train.modes = [
        "execCode(+host, +privilege).", 
        
        "vulExists(+host, -cve, -software).",
        "vulProperty(+cve, #exploitType, #effect).",
        
        "networkServiceInfo(+host, +software, -protocol, +port, -privilege).",
        
        "hacl(+host, -host, -protocol, -port).",
        "hacl(+zone, -host, -protocol, -port).",
        
        "attackerLocated(-host).",
        "attackerLocated(-zone)."

        "initialAccess(-host, -privilege)."
    ]

    return test


# ==========================================
# TESTE
# ==========================================
test = get_mulval_like_data()
probs = clf.predict_proba(test)

print("\n=== TEST RESULTS ===")
print(f"t1 probability (expected HIGH): {probs[0]:.4f}")
print(f"t2 probability (expected LOW):  {probs[1]:.4f}")


=== TEST RESULTS ===
t1 probability (expected HIGH): 0.4574
t2 probability (expected LOW):  0.4574


### Stratified 5-fold

In [ ]:
import os
import numpy as np
from srlearn.rdn import BoostedRDNClassifier
from srlearn import Background, Database

BASE_PATH = "dataset2"
FOLDS = ["fold01", "fold02", "fold03", "fold04", "fold05"]

def merge_files(file_list, output_file):
    with open(output_file, "w") as outfile:
        for fname in file_list:
            with open(fname) as infile:
                outfile.write(infile.read())
                outfile.write("\n")


def load_database(folds, prefix):
    pos_files = []
    neg_files = []
    fact_files = []

    for fold in folds:
        base = os.path.join(BASE_PATH, fold)
        pos_files.append(os.path.join(base, "pos.pl"))
        neg_files.append(os.path.join(base, "neg.pl"))
        fact_files.append(os.path.join(base, "facts.pl"))

    # Arquivos temporários
    merged_pos = f"{prefix}_pos.pl"
    merged_neg = f"{prefix}_neg.pl"
    merged_facts = f"{prefix}_facts.pl"

    merge_files(pos_files, merged_pos)
    merge_files(neg_files, merged_neg)
    merge_files(fact_files, merged_facts)

    db = Database.from_files(merged_pos, merged_neg, merged_facts)

    db.modes = [
        "execCode(+host, +privilege).", 
        "vulExists(+host, -cve, +software).",
        "vulProperty(+cve, #exploitType, #effect).",
        "networkServiceInfo(+host, -software, +protocol, +port, -privilege).",
        "hacl(+host, -host, -protocol, -port).",
        "hacl(+zone, -host, -protocol, -port).",
        "attackerLocated(-host).", # direct on-host access
        "attackerLocated(-zone).", # direct network access
        "initialAccess(-host, -privilege)." # multi-hop access
    ]

    return db


# ==========================================
# 5-FOLD CROSS VALIDATION
# ==========================================

results = []

for i in range(5):
    test_fold = FOLDS[i]
    train_folds = [f for j, f in enumerate(FOLDS) if j != i]

    print(f"\n==============================")
    print(f"Fold {i+1}")
    print(f"Treino: {train_folds}")
    print(f"Teste : {test_fold}")

    # Carregar dados
    train_db = load_database(train_folds, prefix="train_tmp")
    test_db = load_database([test_fold], prefix="test_tmp")

    # Background
    bk = Background(modes=train_db.modes)

    # Modelo
    clf = BoostedRDNClassifier(
        background=bk,
        target="execCode",
        max_tree_depth=4,
        node_size=5,
        n_estimators=75,
    )
    # Treinamento
    clf.fit(train_db)

    probs = clf.predict_proba(test_db)

    import shutil
    from pathlib import Path
    # Diretório criado pelo BoostSRL
    data_dir = Path(clf.file_system.files.DIRECTORY)
    # Pasta de destino no seu projeto
    save_root = Path("results5")
    save_root.mkdir(exist_ok=True)
    save_dir = save_root / f"fold_{i+1}"
    # Remove se já existir
    if save_dir.exists():
        shutil.rmtree(save_dir)
    # Copia tudo
    shutil.copytree(data_dir, save_dir)

    print(f"Modelo salvo em: {save_dir}")

    print("\n=== TEST RESULTS ===")
    print(f"t1 probability: {probs[0]:.4f}")
    print(f"t2 probability:  {probs[1]:.4f}")


Fold 1
Treino: ['fold02', 'fold03', 'fold04', 'fold05']
Teste : fold01
Modelo salvo em: results5/fold_1

=== TEST RESULTS ===
t1 probability: 0.0182
t2 probability:  0.2698

Fold 2
Treino: ['fold01', 'fold03', 'fold04', 'fold05']
Teste : fold02


KeyboardInterrupt: 